In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import gradio as gr

# ========== 資料讀取與合併 ==========
def time_to_seconds(tstr):
    if pd.isnull(tstr): return 0
    parts = str(tstr).split(':')
    if len(parts) == 3:
        h, m, s = int(parts[0]), int(parts[1]), float(parts[2])
        return h * 3600 + m * 60 + s
    elif len(parts) == 2:
        m, s = int(parts[0]), float(parts[1])
        return m * 60 + s
    else:
        try:
            return float(parts[0])
        except:
            return 0

winner = pd.read_csv('./data/winners.csv')
drivers = pd.read_csv('./data/drivers_updated.csv')
teams = pd.read_csv('./data/teams_updated.csv')
laps = pd.read_csv('./data/fastest_laps_updated.csv')

winner['year'] = pd.to_datetime(winner['Date']).dt.year
winner['year_raw'] = winner['year']
winner['Grand Prix raw'] = winner['Grand Prix']

df = winner.merge(
    drivers[['Driver', 'Car', 'year', 'Nationality', 'PTS']],
    left_on=['Winner', 'Car', 'year'],
    right_on=['Driver', 'Car', 'year'],
    how='left',
    suffixes=('', '_driver')
)
df = df.merge(
    laps[['Grand Prix', 'Driver', 'Car', 'year', 'Time']],
    left_on=['Grand Prix', 'Winner', 'Car', 'year'],
    right_on=['Grand Prix', 'Driver', 'Car', 'year'],
    how='left',
    suffixes=('', '_lap')
)
df = df.merge(
    teams[['Team', 'PTS', 'year']],
    left_on=['Car', 'year'],
    right_on=['Team', 'year'],
    how='left',
    suffixes=('', '_team')
)
df['RaceTime_sec'] = df['Time'].apply(time_to_seconds)
df['FastestLap_sec'] = df['Time_lap'].apply(time_to_seconds)
df['year_raw'] = winner['year_raw']
df['Grand Prix raw'] = winner['Grand Prix raw']

cat_cols = ['Winner', 'Car', 'Grand Prix', 'Nationality', 'Team']
num_cols = [
    'Laps', 'PTS', 'PTS_team', 'RaceTime_sec', 'FastestLap_sec', 'year'
]
df['PTS_team'] = df['PTS_team'].fillna(0)
df[num_cols] = df[num_cols].fillna(0)

value_counts = df['Winner'].value_counts()
valid_drivers = value_counts[value_counts >= 2].index
df = df[df['Winner'].isin(valid_drivers)].reset_index(drop=True)

X_cat = df[cat_cols].astype(str).values
X_num = df[num_cols].values
y = df['Winner'].astype(str).values
df['orig_index'] = df.index

# ========== 分割資料 ==========
X_cat_train, X_cat_test, X_num_train, X_num_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_cat, X_num, y, df['orig_index'].values, test_size=0.2, random_state=42, stratify=y
)

# ========== Open-set LabelEncoder ==========
class SafeLabelEncoder:
    def __init__(self):
        self.le = LabelEncoder()
        self.classes_ = None
        self.class2idx = None
    def fit(self, X):
        self.le.fit(X)
        self.classes_ = list(self.le.classes_) + ['__UNK__']
        self.class2idx = {c: i for i, c in enumerate(self.classes_)}
    def transform(self, X):
        return np.array([self.class2idx.get(x, self.class2idx['__UNK__']) for x in X], dtype=np.int64)
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)
    def inverse_transform(self, X):
        return np.array([self.classes_[i] for i in X])

encoders = {}
for i, col in enumerate(cat_cols):
    le = SafeLabelEncoder()
    X_cat_train[:, i] = le.fit_transform(X_cat_train[:, i])
    X_cat_test[:, i] = le.transform(X_cat_test[:, i])
    encoders[col] = le

le_winner = encoders['Winner']
y_train_enc = le_winner.transform(y_train)
y_test_enc = le_winner.transform(y_test)

scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train).astype(np.float32)
X_num_test = scaler.transform(X_num_test).astype(np.float32)
y_train_enc = y_train_enc.astype(np.int64)
y_test_enc = y_test_enc.astype(np.int64)
X_cat_train = X_cat_train.astype(np.int64)
X_cat_test = X_cat_test.astype(np.int64)

# ========== PyTorch Dataset ==========
class F1RaceSet(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X_cat[idx], self.X_num[idx], self.y[idx]

batch_size = 128
trainset = F1RaceSet(X_cat_train, X_num_train, y_train_enc)
testset = F1RaceSet(X_cat_test, X_num_test, y_test_enc)
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(testset, batch_size=batch_size)

# ========== Model ==========
class F1DNN(nn.Module):
    def __init__(self, cat_dims, num_num_features, embedding_dim=8, hidden_dim=128, num_classes=None):
        super().__init__()
        self.emb_layers = nn.ModuleList([
            nn.Embedding(cat_dim, embedding_dim) for cat_dim in cat_dims
        ])
        input_dim = embedding_dim * len(cat_dims) + num_num_features
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.emb_layers)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.mlp(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cat_dims = [len(encoders[col].classes_) for col in cat_cols]
num_classes = len(encoders['Winner'].classes_)
model = F1DNN(cat_dims, len(num_cols), embedding_dim=8, hidden_dim=128, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ========== 訓練與測試集評估 ==========
epochs = 30
OPENSET_THRESHOLD = 0.7

for epoch in range(epochs):
    # Training
    model.train()
    total_loss = 0
    for X_cat_batch, X_num_batch, y_batch in train_loader:
        X_cat_batch, X_num_batch, y_batch = X_cat_batch.to(device), X_num_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_cat_batch, X_num_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
    avg_loss = total_loss / len(trainset)

    # Test set evaluation
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for X_cat_batch, X_num_batch, y_batch in test_loader:
            X_cat_batch, X_num_batch, y_batch = X_cat_batch.to(device), X_num_batch.to(device), y_batch.to(device)
            output = model(X_cat_batch, X_num_batch)
            loss = criterion(output, y_batch)
            test_loss += loss.item() * len(y_batch)
            preds = torch.argmax(output, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    avg_test_loss = test_loss / total
    test_acc = correct / total

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_loss:.4f}, Test Loss: {avg_test_loss:.4f}, Test Acc: {test_acc:.4f}")

print("\n==== Final Test Set Evaluation ====")
print(f"Test set — Avg Loss: {avg_test_loss:.4f}  |  Accuracy: {test_acc:.4f}")

# ========== open-set 預測函數 ==========
def openset_predict(model, X_cat, X_num, encoders, threshold=OPENSET_THRESHOLD):
    model.eval()
    unknown_mask = np.any(X_cat == encoders['Winner'].class2idx['__UNK__'], axis=1)
    X_cat_tensor = torch.tensor(X_cat, dtype=torch.long).to(device)
    X_num_tensor = torch.tensor(X_num, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_cat_tensor, X_num_tensor)
        probs = nn.functional.softmax(logits, dim=1).cpu().numpy()
        max_conf = probs.max(axis=1)
        preds = probs.argmax(axis=1)
    final_preds = []
    for i, (p, conf, unk) in enumerate(zip(preds, max_conf, unknown_mask)):
        if unk or conf < threshold:
            final_preds.append('Unknown')
        else:
            final_preds.append(encoders['Winner'].classes_[p])
    return final_preds, max_conf

# ====== Gradio 互動介面，支援未來賽事與測試集查詢 ======
all_years = sorted(df['year_raw'].unique())
future_years = [y for y in range(all_years[-1] + 1, all_years[-1] + 3)]
all_years_extended = all_years + future_years
all_gps = sorted(df['Grand Prix raw'].unique())

test_df = df.iloc[idx_test].reset_index(drop=True)
test_years = sorted(test_df['year_raw'].unique())
test_gps_by_year = {y: sorted(test_df[test_df['year_raw']==y]['Grand Prix raw'].unique()) for y in test_years}

def get_prev_season_driverlist(year):
    last_year = year - 1
    driver_groups = drivers[drivers['year'] == last_year].copy()
    driver_groups = driver_groups[['Driver', 'Car', 'Nationality', 'PTS']].drop_duplicates()
    team_pts = teams[teams['year'] == last_year][['Team', 'PTS']].rename(columns={'Team': 'Car', 'PTS': 'PTS_team'})
    driver_groups = driver_groups.merge(team_pts, on='Car', how='left')
    driver_groups['PTS_team'] = driver_groups['PTS_team'].fillna(0)
    driver_groups = driver_groups.rename(columns={'Driver': 'Winner'})
    return driver_groups
def get_real_participants(year, grand_prix=None):
    # 忽略分站，只用全年名單
    if 'Driver' not in drivers.columns:
        print("drivers.csv 檔案缺少 Driver 欄位！")
        return []
    rows = drivers[drivers['year'] == year]
    return rows['Driver'].unique().tolist()


def predict_future_all(year, grand_prix):
    real_participants = get_real_participants(year)
    driver_list = get_prev_season_driverlist(year)
    driver_list = driver_list[driver_list['Winner'].isin(real_participants)]

    if driver_list is None or driver_list.empty:
        return f"查無 {year} {grand_prix} 所有參賽車手資料，無法預測。"
    X_cat_fut, X_num_fut, driver_names, car_names = [], [], [], []
    for _, row in driver_list.iterrows():
        feat = {
            'Winner': row['Winner'],
            'Car': row['Car'],
            'Grand Prix': grand_prix,
            'Nationality': row['Nationality'],
            'Team': row['Car']  # 假設 Team=Car
        }
        X_cat_fut.append([feat[c] for c in cat_cols])
        X_num_fut.append([year, row['PTS'], row['PTS_team'], 0, 0, year])
        driver_names.append(row['Winner'])
        car_names.append(row['Car'])
    X_cat_fut = np.array(X_cat_fut)
    X_num_fut = np.array(X_num_fut, dtype=np.float32)
    for i, col in enumerate(cat_cols):
        X_cat_fut[:, i] = encoders[col].transform(X_cat_fut[:, i])
    X_cat_fut = X_cat_fut.astype(np.int64)
    X_num_fut = scaler.transform(X_num_fut).astype(np.float32)
    fut_preds, fut_conf = openset_predict(model, X_cat_fut, X_num_fut, encoders, threshold=OPENSET_THRESHOLD)
    fut_results = []
    for i, (pred, conf, driver, car) in enumerate(zip(fut_preds, fut_conf, driver_names, car_names)):
        fut_results.append((conf, pred, driver, car))
    fut_results = sorted(fut_results, reverse=True, key=lambda x: x[0])
    out_txt = f"【{year} {grand_prix} 奪冠 open-set 預測（限實際參賽者）】\n"
    for i, (conf, pred, driver, car) in enumerate(fut_results[:10]):
        out_txt += f"{i+1}. {driver}（車隊：{car}）→ 預測：{pred}，機率/信心：{conf:.3f}\n"
    return out_txt

def predict_testset(year, grand_prix):
    rows = test_df[(test_df['year_raw']==year) & (test_df['Grand Prix raw']==grand_prix)]
    if rows.empty:
        return f"{year} {grand_prix} 在測試集找不到。"
    X_cat_query = []
    X_num_query = []
    true_names = []
    for _, row in rows.iterrows():
        idx = row.name
        X_cat_query.append(X_cat_test[idx])
        X_num_query.append(X_num_test[idx])
        true_names.append(row['Winner'])
    X_cat_query = np.vstack(X_cat_query).astype(np.int64)
    X_num_query = np.vstack(X_num_query).astype(np.float32)
    preds, confs = openset_predict(model, X_cat_query, X_num_query, encoders, threshold=OPENSET_THRESHOLD)
    out_txt = f"【{year} {grand_prix} 測試集 open-set 預測】\n"
    for i, (true_winner, pred, conf) in enumerate(zip(true_names, preds, confs)):
        out_txt += f"真實冠軍：{true_winner}｜模型預測：{pred}｜信心：{conf:.3f}\n"
    return out_txt

def interface_selector(mode, year, grand_prix):
    if mode == "未來賽事預測":
        return predict_future_all(year, grand_prix)
    else:
        return predict_testset(year, grand_prix)

def get_gp_choices(mode, year):
    if mode == "未來賽事預測":
        return gr.update(choices=all_gps)
    else:
        if year in test_gps_by_year:
            return gr.update(choices=test_gps_by_year[year])
        else:
            return gr.update(choices=[])

with gr.Blocks() as demo:
    gr.Markdown("### F1 Open-Set 冠軍預測（未來模擬／測試集查詢）")
    mode_dropdown = gr.Dropdown(choices=["未來賽事預測", "測試集查詢"], label="模式")
    year_dropdown = gr.Dropdown(choices=all_years_extended, label="年份")
    gp_dropdown = gr.Dropdown(choices=all_gps, label="分站名（Grand Prix）")
    out_box = gr.Textbox(label="open-set 預測結果", lines=12)

    mode_dropdown.change(get_gp_choices, [mode_dropdown, year_dropdown], gp_dropdown)
    year_dropdown.change(get_gp_choices, [mode_dropdown, year_dropdown], gp_dropdown)
    btn = gr.Button("預測/查詢")
    btn.click(interface_selector, [mode_dropdown, year_dropdown, gp_dropdown], out_box)

demo.launch()




Epoch 1/30, Train Loss: 4.3335, Test Loss: 4.2171, Test Acc: 0.1395
Epoch 2/30, Train Loss: 3.7393, Test Loss: 3.8746, Test Acc: 0.3628
Epoch 3/30, Train Loss: 3.2959, Test Loss: 3.4254, Test Acc: 0.5488
Epoch 4/30, Train Loss: 2.9642, Test Loss: 2.9593, Test Acc: 0.6558
Epoch 5/30, Train Loss: 2.6218, Test Loss: 2.5447, Test Acc: 0.6837
Epoch 6/30, Train Loss: 2.3609, Test Loss: 2.2146, Test Acc: 0.7302
Epoch 7/30, Train Loss: 2.1255, Test Loss: 1.9404, Test Acc: 0.7535
Epoch 8/30, Train Loss: 1.9233, Test Loss: 1.7246, Test Acc: 0.7860
Epoch 9/30, Train Loss: 1.7350, Test Loss: 1.5478, Test Acc: 0.7953
Epoch 10/30, Train Loss: 1.5487, Test Loss: 1.3930, Test Acc: 0.8186
Epoch 11/30, Train Loss: 1.4074, Test Loss: 1.2635, Test Acc: 0.8186
Epoch 12/30, Train Loss: 1.2572, Test Loss: 1.1457, Test Acc: 0.8372
Epoch 13/30, Train Loss: 1.1894, Test Loss: 1.0391, Test Acc: 0.8558
Epoch 14/30, Train Loss: 1.0822, Test Loss: 0.9424, Test Acc: 0.8605
Epoch 15/30, Train Loss: 0.9614, Test Loss: